# Análise de Consumo e Pedidos Scada-BPS

#### Calcular quantidade de dias de um intervalo:

In [27]:
from datetime import date, datetime

def calcular_dias_entre_datas(data_inicio_str, data_fim_str, formato="%d/%m/%Y"):
    """
    Calcula o número de dias entre duas datas.

    Args:
        data_inicio_str (str): A data de início no formato de string (ex: "01/01/2023").
        data_fim_str (str): A data final no formato de string (ex: "31/12/2023").
        formato (str): O formato da data utilizada nas strings. Padrão é "%d/%m/%Y".

    Returns:
        int: A diferença absoluta em dias entre as duas datas.
    """
    try:
        # Converte as strings de data para objetos datetime.date
        # O strptime() converte a string para datetime, e o .date() extrai apenas a data.
        data_inicio = datetime.strptime(data_inicio_str, formato).date()
        data_fim = datetime.strptime(data_fim_str, formato).date()

        # Calcula a diferença (o resultado é um objeto timedelta)
        diferenca = data_fim - data_inicio

        # Retorna o número de dias da diferença
        # Usamos abs() para garantir que o resultado seja sempre positivo,
        # independentemente da ordem das datas.
        return abs(diferenca.days)

    except ValueError as e:
        print(f"Erro ao converter a data. Verifique o formato '{formato}' e o valor das datas.")
        print(f"Detalhes do erro: {e}")
        return -1 # Retorna um valor de erro

# --- Exemplo de Uso ---
if __name__ == "__main__":
    # Defina as datas que você deseja calcular
    data_1 = "17/10/2025"
    data_2 = "17/11/2025"

    # Define o formato esperado (Dia/Mês/Ano)
    formato_data = "%d/%m/%Y"

    # Chama a função para calcular
    quantidade_dias = calcular_dias_entre_datas(data_1, data_2, formato_data) + 1

    if quantidade_dias != -1:
        print("-" * 40)
        print("CÁLCULO DE INTERVALO DE DATAS")
        print("-" * 40)
        print(f"Data de Início: {data_1}")
        print(f"Data Final: {data_2}")
        print(f"\nA quantidade de dias entre as duas datas é: {quantidade_dias} dias.")
        print("-" * 40)

    # Exemplo com datas em outra ordem
    data_3 = "10/01/2025"
    data_4 = "01/01/2025"
    dias_reverso = calcular_dias_entre_datas(data_3, data_4, formato_data)
    if dias_reverso != -1:
        print(f"Teste de ordem reversa ({data_3} a {data_4}): {dias_reverso} dias.")
        print("-" * 40)

    print(quantidade_dias)

----------------------------------------
CÁLCULO DE INTERVALO DE DATAS
----------------------------------------
Data de Início: 17/10/2025
Data Final: 17/11/2025

A quantidade de dias entre as duas datas é: 32 dias.
----------------------------------------
Teste de ordem reversa (10/01/2025 a 01/01/2025): 9 dias.
----------------------------------------
32


### Importação bases

In [28]:
import pandas as pd

consumo_df = pd.read_excel("base_consumo_bps.xlsx")
contagem_df = pd.read_excel("base_contagem_semanal.xlsx")



### Tratamento dos dados

In [29]:
# Acrescentar coluna com cálculo de consumo médio diário:
consumo_df['Consumo Médio Dia'] = consumo_df['Qtde.'] / quantidade_dias
# Acrescentar coluna com a necessidade de estoque. inserir a quantidade de dias para suprimento e o percentual de segurança:
consumo_df['Necessidade Estoque'] = (consumo_df['Consumo Médio Dia'] * 7) * 1.15

# Arredondar as colunas numéricas:
colunas_arredondar = ["Qtde.", "Consumo Médio Dia", "Necessidade Estoque"]
consumo_df[colunas_arredondar] = consumo_df[colunas_arredondar].round(0)

consumo_df.info()
consumo_df.head(20)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 158 entries, 0 to 157
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Cód. Item            145 non-null    object 
 1   Item                 133 non-null    object 
 2   Unidade              146 non-null    object 
 3   Qtde.                146 non-null    float64
 4   Consumo Médio Dia    146 non-null    float64
 5   Necessidade Estoque  146 non-null    float64
dtypes: float64(3), object(3)
memory usage: 7.5+ KB


,Cód. Item,Item,Unidade,Qtde.,Consumo Médio Dia,Necessidade Estoque
0,BEBIDAS,NaN,NaN,NaN,NaN,NaN
1,57,AGUA COM GAS CIFAO,LT,125.0,4.0,32.0
2,186,AGUA GARRAFA C GAS,UN,402.0,13.0,101.0
3,185,AGUA GARRAFA S GAS,UN,427.0,13.0,107.0
4,188,AGUA TONICA LATA,UN,15.0,0.0,4.0
5,40,AROMATIZANTE AVELA,LT,1.0,0.0,0.0
6,42,AROMATIZANTE FRAMBOESA,LT,1.0,0.0,0.0
7,43,AROMATIZANTE MACA VERDE,LT,1.0,0.0,0.0
8,485,CERVEJA HEINEKEN LG NECK,UN,19.0,1.0,5.0
9,177,COCA COLA LATA,UN,80.0,2.0,20.0


In [31]:

contagem_df.info()
contagem_df.head(20)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 153 entries, 0 to 152
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   EST                      152 non-null    float64
 1   L J                      152 non-null    float64
 2   E+L
unid                 144 non-null    object 
 3   Fornecedor               150 non-null    object 
 4   Produto                  150 non-null    object 
 5   Emb                      149 non-null    object 
 6   Qtd/Emb                  147 non-null    object 
 7   Total                    153 non-null    int64  
 8   Pedido                   152 non-null    float64
 9   Vendido (Degust)         153 non-null    int64  
 10  Ped                      153 non-null    int64  
 11  Unnamed: 11              0 non-null      float64
 12  Relatório Degust         87 non-null     float64
 13  Unnamed: 13              0 non-null      float64
 14  Pedido Pendente          0

,EST,L J,E+L\nunid,Fornecedor,Produto,Emb,Qtd/Emb,Total,Pedido,Vendido (Degust),Ped,Unnamed: 11,Relatório Degust,Unnamed: 13,Pedido Pendente,Unnamed: 15,Unnamed: 16,Variável produto,Unnamed: 18,Validação Cadastro item
0,0.0,0.0,-,BOTAFOGO E GELO,AGUA TONICA LATA,CX,12,0,NaN,0,0,NaN,3.0000,NaN,NaN,0,NaN,NaN,NaN,188
1,0.0,0.0,-,BOTAFOGO E GELO,CERVEJA HEINEKEN LG NECK,CX,24,0,0.0,0,0,NaN,4.0000,NaN,NaN,0,NaN,NaN,NaN,485
2,0.0,0.0,-,BOTAFOGO E GELO,COCA COLA LATA,CX,12,0,0.0,2,0,NaN,21.0000,NaN,NaN,2,NaN,NaN,NaN,177
3,0.0,0.0,-,BOTAFOGO E GELO,COCA COLA ZERO LATA,CX,12,0,0.0,3,0,NaN,41.0000,NaN,NaN,3,NaN,NaN,NaN,178
4,0.0,0.0,-,BOTAFOGO E GELO,GUARANA LATA,CX,12,0,0.0,0,0,NaN,5.0000,NaN,NaN,0,NaN,NaN,NaN,182
5,0.0,0.0,-,BOTAFOGO E GELO,GUARANA ZERO LATA,CX,12,0,0.0,0,0,NaN,3.0000,NaN,NaN,0,NaN,NaN,NaN,183
6,0.0,0.0,-,UNIQUE - PEDIDO TODO DIA 15,BLEND 1KG,PCT,1,0,0.0,10,0,NaN,10.4906,NaN,NaN,10,NaN,NaN,NaN,1149
7,0.0,0.0,-,UNIQUE - PEDIDO TODO DIA 15,BLEND 250G,PCT,0.25,0,0.0,7,0,NaN,1.8400,NaN,NaN,7,NaN,NaN,NaN,662
8,0.0,0.0,-,UNIQUE - PEDIDO TODO DIA 15,B. VERMELHO 250G,PCT,0.25,0,0.0,7,0,NaN,1.7000,NaN,NaN,7,NaN,NaN,NaN,723
9,0.0,0.0,-,UNIQUE - PEDIDO TODO DIA 15,B. AMARELO 250G,PCT,0.25,0,0.0,1,0,NaN,0.3100,NaN,NaN,1,NaN,NaN,NaN,725
